# Text-to-SQL with RAG: Local LLM + SQL Server

Ask questions about a SQL Server database in plain English and get back the SQL, the results and a written analysis. Everything is generated by an open-source LLM running **entirely on your own machine**.

> *"What is the total revenue for premium customers last quarter, and how does it compare with the previous quarter?"*

No data leaves your machine. The model runs locally through [Ollama](https://ollama.com), and the embeddings run locally through `sentence-transformers`.

## How it works

```mermaid
flowchart LR
    Q[User question] --> R[1. Retrieve business rules<br/>from data dictionary]
    R --> G[2. LLM writes T-SQL<br/>using rules + schema]
    G --> X[3. Run on SQL Server]
    X -->|error| G
    X -->|rows| E[4. LLM explains results]
    E --> A[Answer + SQL + data]
```

The steps run in a fixed order. The code decides *what happens next* and the LLM does only two focused jobs: writing one SQL query and explaining the results. If the query fails, the error is sent back to the model so it can fix the query, up to three attempts.

### Why a fixed pipeline rather than an autonomous agent?

The first version of this project used a LangChain agent that chose its own tools. With a 7B local model, it often repeated tool calls, wrote tool calls as plain text, or stopped with an empty answer partway through. A fixed pipeline gives the model far less room to go wrong and makes every step easy to inspect, which suits small local models.

It also means the model doesn't need tool calling at all, so the project can use `qwen2.5-coder:7b`. That model is trained heavily on code (including text-to-SQL) and writes noticeably better SQL than the general `qwen2.5:7b`.

| Component | Tool |
|---|---|
| LLM | `qwen2.5-coder:7b` served by Ollama, called via `langchain-ollama` |
| RAG | `all-MiniLM-L6-v2` embeddings + Chroma vector store |
| Database | SQL Server Express, connected with `pyodbc` + SQLAlchemy |

## Prerequisites

| Requirement | Notes |
|---|---|
| **Python 3.12** | LangChain 0.3 does not support Python 3.14 |
| **Ollama** | Install from [ollama.com/download](https://ollama.com/download), then run `ollama pull qwen2.5-coder:7b` |
| **SQL Server Express** | This notebook uses the named instance `localhost\SQLEXPRESS` |
| **ODBC Driver 17 or 18 for SQL Server** | Required by `pyodbc` |
| **Demo database** | Run `create_TextToSQLDemo.sql` in SSMS to create and populate `TextToSQLDemo` |

Install the Python packages in a Python 3.12 virtual environment:

```bash
pip install -r requirements.txt
```

## 1. Configuration

Everything you might want to change lives in this one cell.

In [17]:
from datetime import date
from pathlib import Path
import re
import urllib.parse

import pyodbc
from langchain_community.utilities import SQLDatabase
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama

# --- Settings ---------------------------------------------------------
SERVER = r"localhost\SQLEXPRESS"     # SQL Server instance
DATABASE = "TextToSQLDemo"            # database created by create_TextToSQLDemo.sql
MODEL = "qwen2.5-coder:7b"            # code model: much stronger at writing SQL
NUM_CTX = 8192                        # context window in tokens (Ollama's default is too small)
EMBEDDING_MODEL = "all-MiniLM-L6-v2"  # small, fast local embedding model
DICTIONARY_FILE = Path("data_dictionary.txt")
MAX_SQL_ATTEMPTS = 3                  # how many times the model may fix a failing query

assert DICTIONARY_FILE.exists(), f"Can't find {DICTIONARY_FILE.resolve()}"
print("Configuration loaded.")

Configuration loaded.


## 2. Connect to SQL Server

The connection uses **Windows authentication** (no username or password in the code) and picks the newest installed ODBC driver automatically.

`TrustServerCertificate=yes` is needed because a local SQL Server Express install uses a self-signed certificate.

In [18]:
drivers = [d for d in pyodbc.drivers() if "ODBC Driver" in d and "SQL Server" in d]
if not drivers:
    raise RuntimeError("No 'ODBC Driver for SQL Server' found. Install ODBC Driver 17 or 18 from Microsoft.")
driver = sorted(drivers)[-1]

conn_str = (
    f"Driver={{{driver}}};"
    f"Server={SERVER};"
    f"Database={DATABASE};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)
db = SQLDatabase.from_uri(
    "mssql+pyodbc:///?odbc_connect=" + urllib.parse.quote_plus(conn_str),
    sample_rows_in_table_info=2,   # include 2 example rows per table in the schema
)

# The schema is small and doesn't change, so read it once and reuse it.
SCHEMA = db.get_table_info()

print(f"Connected using {driver}")
print("Tables:", db.get_usable_table_names())

Connected using ODBC Driver 18 for SQL Server
Tables: ['Customers', 'OrderItems', 'Orders', 'Products']


## 3. Load the local LLM

- `temperature=0` makes the model as deterministic as possible, which is what you want when it's writing SQL.
- `num_ctx` sets the context window. Ollama's default is too small once the schema, rules and results are included. Ollama then silently drops the start of the prompt.

In [19]:
llm = ChatOllama(model=MODEL, temperature=0, num_ctx=NUM_CTX)
# Retries use a little randomness so the model doesn't repeat the exact same mistake.
llm_retry = ChatOllama(model=MODEL, temperature=0.4, num_ctx=NUM_CTX)
print(llm.invoke("Reply with the single word: ready").content)

ready


## 4. Build the RAG retriever over the data dictionary

The database schema tells the model *what columns exist*, but not *what the business means*. For example, it can't know that revenue excludes cancelled orders, or that a "premium" customer has spent over £10,000. Those rules live in `data_dictionary.txt`.

The dictionary is split into chunks, embedded and stored in Chroma. For each question, the most relevant rules are retrieved and passed to the model.

Each chunk gets a fixed ID, so re-running this cell **updates** the store instead of adding duplicate copies.

In [20]:
docs = TextLoader(str(DICTIONARY_FILE), encoding="utf-8").load()
splits = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL),
    collection_name="data_dictionary",
    ids=[f"chunk-{i}" for i in range(len(splits))],
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


def retrieve_rules(question: str) -> str:
    """Return the dictionary entries most relevant to the question."""
    return "\n\n".join(doc.page_content for doc in retriever.invoke(question))


print(f"Indexed {len(splits)} chunks from {DICTIONARY_FILE.name}")
print("\nExample retrieval for 'premium customers':\n")
print(retrieve_rules("premium customers")[:400], "...")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 51125.84it/s]


Indexed 8 chunks from data_dictionary.txt

Example retrieval for 'premium customers':

BUSINESS RULE: PREMIUM CUSTOMERS
A premium customer is a customer whose lifetime revenue (all Completed orders, any date) is greater than 10,000. Calculate it with GROUP BY o.CustomerID HAVING SUM(oi.Quantity * oi.UnitPrice * (1 - oi.Discount)) > 10000. Premium status is based on lifetime spend, not spend within the period being analysed.

TABLE: dbo.Products
One row per product. Columns: ProductI ...


## 5. Prompts

There are two prompts, one for each job the LLM does:

1. **SQL prompt:** turns the question, business rules and schema into a single T-SQL query.
2. **Explanation prompt:** turns the question, SQL and results into a written answer.

Small models are unreliable at date arithmetic: asked for "last quarter", a 7B model will often pick the wrong one. So `date_context()` works out the exact quarter boundaries in Python and passes them to the model as ready-made filters.

These boundaries are only included when the question mentions a relative period ("last quarter", "this year", …). Otherwise small models tend to add date filters the question never asked for.

The SQL prompt also includes two **worked examples** (few-shot prompting). They use different questions from the demos, but show the patterns the model should copy: the revenue formula, the `Completed` filter, quarter date ranges, and premium status based on lifetime spend.

In [21]:
RELATIVE_PERIOD = re.compile(
    r"\b(last|previous|prior|this|current|recent|latest|quarter|quarterly|ytd|year to date)\b",
    re.IGNORECASE,
)


def date_context(question: str = "", today: date | None = None) -> str:
    """Exact date boundaries for relative periods, so the LLM doesn't have to calculate them."""
    today = today or date.today()
    if not RELATIVE_PERIOD.search(question):
        return (f"Today: {today.isoformat()}. The question does not ask about a relative period, "
                "so do NOT add any date filter unless the question names a specific date or year.")

    def quarter_start(d: date, offset: int = 0) -> date:
        m = (d.month - 1) // 3 * 3 + 3 * offset   # months from January of d's year
        return date(d.year + m // 12, m % 12 + 1, 1)

    def label(d: date) -> str:
        return f"{d.year} Q{(d.month - 1) // 3 + 1}"

    cur, last, prev = quarter_start(today), quarter_start(today, -1), quarter_start(today, -2)
    return (
        f"Today: {today.isoformat()}\n"
        f"Current quarter (not finished): {label(cur)}, starts {cur}\n"
        f"Last quarter: {label(last)} -> o.OrderDate >= '{last}' AND o.OrderDate < '{cur}'\n"
        f"Previous quarter: {label(prev)} -> o.OrderDate >= '{prev}' AND o.OrderDate < '{last}'\n"
        f"This year: {today.year}. Last year: {today.year - 1}."
    )


print(date_context("Revenue last quarter vs the previous quarter?"))
print()
print(date_context("Which year had the highest revenue?"))


SQL_PROMPT = """You are an expert Microsoft SQL Server developer.
Write ONE T-SQL query that answers the question.

Dates:
{dates}

Business rules (apply these EXACTLY: formulas, status filters, definitions, date boundaries):
{rules}

Database schema:
{schema}

Requirements:
- T-SQL only (use TOP, not LIMIT). Read-only: a single SELECT, optionally starting with WITH.
- SQL Server syntax only. Do NOT use PostgreSQL/MySQL syntax: no FILTER (WHERE ...), no :: casts,
  no DATE_TRUNC, no INTERVAL, no LIMIT, no NULLS LAST. Use CAST(... AS DECIMAL(12,2)) for casts
  and CASE WHEN for conditional sums.
- To limit rows use SELECT TOP n (e.g. SELECT TOP 1 ... ORDER BY Revenue DESC).
  Never use OFFSET or FETCH.
- Keep it simple: prefer one GROUP BY over correlated subqueries, and never put an aggregate
  inside another aggregate.
- Only filter by date when the question asks about a period. Never add filters the question
  didn't ask for.
- Use only tables and columns that appear in the schema.
- Give every calculated column a clear alias.
- Return ONLY the SQL inside a ```sql code block, with no explanation.

Example 1
Question: What was revenue from London customers in each of the last two quarters?
```sql
SELECT YEAR(o.OrderDate) AS [Year], DATEPART(QUARTER, o.OrderDate) AS [Quarter],
       SUM(oi.Quantity * oi.UnitPrice * (1 - oi.Discount)) AS Revenue
FROM dbo.Orders o
JOIN dbo.OrderItems oi ON oi.OrderID = o.OrderID
JOIN dbo.Customers c   ON c.CustomerID = o.CustomerID
WHERE o.Status = 'Completed'
  AND c.Region = 'London'
  AND o.OrderDate >= '<previous quarter start>' AND o.OrderDate < '<current quarter start>'
GROUP BY YEAR(o.OrderDate), DATEPART(QUARTER, o.OrderDate)
ORDER BY [Year], [Quarter]
```

Example 2
Question: How many premium customers are there in each region?
```sql
WITH premium AS (
    SELECT o.CustomerID
    FROM dbo.Orders o
    JOIN dbo.OrderItems oi ON oi.OrderID = o.OrderID
    WHERE o.Status = 'Completed'
    GROUP BY o.CustomerID
    HAVING SUM(oi.Quantity * oi.UnitPrice * (1 - oi.Discount)) > 10000
)
SELECT c.Region, COUNT(*) AS PremiumCustomers
FROM premium p
JOIN dbo.Customers c ON c.CustomerID = p.CustomerID
GROUP BY c.Region
ORDER BY PremiumCustomers DESC
```
Example 3
Question: Which region had the highest revenue last year?
```sql
SELECT TOP 1 c.Region,
       SUM(oi.Quantity * oi.UnitPrice * (1 - oi.Discount)) AS Revenue
FROM dbo.Orders o
JOIN dbo.OrderItems oi ON oi.OrderID = o.OrderID
JOIN dbo.Customers c   ON c.CustomerID = o.CustomerID
WHERE o.Status = 'Completed'
  AND o.OrderDate >= '<last year>-01-01' AND o.OrderDate < '<this year>-01-01'
GROUP BY c.Region
ORDER BY Revenue DESC
```

Note: premium status is ALWAYS decided on lifetime spend (a CTE like Example 2, no date filter),
then joined to the orders for the period being asked about.
{error_feedback}
Question: {question}
"""

EXPLAIN_PROMPT = """You are a data analyst. Answer the question using ONLY the query results below.
Currency is GBP (£).

Dates:
{dates}

Question: {question}

SQL that was run:
{sql}

Query results:
{results}

Write a clear, concise answer: state the key figures, what they show, and any trend or
comparison the question asks for (with % change where relevant). Do not invent numbers.
"""

Today: 2026-11-08
Current quarter (not finished): 2026 Q4, starts 2026-10-01
Last quarter: 2026 Q3 -> o.OrderDate >= '2026-07-01' AND o.OrderDate < '2026-10-01'
Previous quarter: 2026 Q2 -> o.OrderDate >= '2026-04-01' AND o.OrderDate < '2026-07-01'
This year: 2026. Last year: 2025.

Today: 2026-11-08. The question does not ask about a relative period, so do NOT add any date filter unless the question names a specific date or year.


## 6. Pipeline functions

- `extract_sql` pulls the query out of the model's reply, then `repair_tsql` fixes common small-model mistakes that SQL Server rejects (`TOP`/`LIMIT` at the end of the query, `FETCH` without `OFFSET`).
- `is_read_only` is a basic safety check that blocks anything other than a `SELECT`. You should still connect with a read-only login.
- `generate_and_run_sql` writes the query, runs it, and if SQL Server returns an error, sends the error back to the model to fix it.

In [22]:
BLOCKED = re.compile(
    r"\b(INSERT|UPDATE|DELETE|MERGE|DROP|ALTER|CREATE|TRUNCATE|EXEC|EXECUTE|GRANT|REVOKE)\b",
    re.IGNORECASE,
)


def main_select_end(sql: str) -> int | None:
    """Position just after the outermost (final) SELECT, skipping CTEs, subqueries and strings."""
    depth, in_string, last = 0, False, None
    for m in re.finditer(r"'|\(|\)|\bSELECT\b", sql, re.I):
        token = m.group(0)
        if token == "'":
            in_string = not in_string
        elif in_string:
            continue
        elif token == "(":
            depth += 1
        elif token == ")":
            depth -= 1
        elif depth == 0:
            last = m.end()
    if last is not None:
        distinct = re.match(r"\s+DISTINCT\b", sql[last:], re.I)
        if distinct:
            last += distinct.end()
    return last


def repair_tsql(sql: str) -> str:
    """Fix common small-model mistakes that SQL Server rejects."""
    # "... ORDER BY x DESC TOP 1" or "... LIMIT 1"  ->  "SELECT TOP 1 ... ORDER BY x DESC"
    trailing = re.search(r"\s+(?:TOP|LIMIT)\s+(\d+)\s*$", sql, re.I)
    if trailing:
        sql = sql[:trailing.start()]
        pos = main_select_end(sql)
        if pos is not None:
            sql = f"{sql[:pos]} TOP {trailing.group(1)}{sql[pos:]}"
    # SQL Server only allows FETCH after OFFSET.
    if re.search(r"\bFETCH\s+(FIRST|NEXT)\b", sql, re.I) and not re.search(r"\bOFFSET\b", sql, re.I):
        sql = re.sub(r"\bFETCH\s+(FIRST|NEXT)\b", "OFFSET 0 ROWS FETCH NEXT", sql, count=1, flags=re.I)
    return sql


def extract_sql(text: str) -> str:
    """Take the SQL from a ```sql block, or fall back to the whole reply."""
    match = re.search(r"```(?:sql)?\s*(.*?)```", text, re.DOTALL | re.IGNORECASE)
    sql = (match.group(1) if match else text).strip()
    # Drop leading comments and semicolons (e.g. ";WITH ..." is common T-SQL style)
    sql = re.sub(r"^(\s*(--[^\n]*\n|/\*.*?\*/|;))+", "", sql, flags=re.DOTALL)
    return repair_tsql(sql.strip().rstrip(";").strip())


def is_read_only(sql: str) -> bool:
    first_word = sql.lstrip().split(None, 1)[0].upper() if sql.strip() else ""
    return first_word in {"SELECT", "WITH"} and not BLOCKED.search(sql)


def generate_and_run_sql(question: str, rules: str) -> tuple[str, str]:
    """Ask the LLM for SQL, run it, and let it fix errors. Returns (sql, results)."""
    error_feedback = ""
    for attempt in range(1, MAX_SQL_ATTEMPTS + 1):
        model = llm if attempt == 1 else llm_retry
        reply = model.invoke(SQL_PROMPT.format(
            dates=date_context(question),
            rules=rules,
            schema=SCHEMA,
            error_feedback=error_feedback,
            question=question,
        )).content
        sql = extract_sql(reply)

        if not is_read_only(sql):
            print(f"Attempt {attempt} rejected (not a single read-only SELECT):\n{sql[:500]}\n")
            error_feedback = "\nYour previous answer was not a single read-only SELECT query. Try again.\n"
            continue

        try:
            return sql, db.run(sql, include_columns=True)
        except Exception as e:
            error = str(e).split("\n")[0][:500]
            print(f"Attempt {attempt} failed: {error}\n--- SQL tried ---\n{sql}\n")
            error_feedback = (
                f"\nYour previous query failed.\nQuery:\n{sql}\nError:\n{error}\n"
                "Fix the error and write a corrected query. Do not repeat the same query.\n"
            )
    raise RuntimeError(
        f"Could not produce a working query after {MAX_SQL_ATTEMPTS} attempts. Last attempt:\n{sql}"
    )

## 7. Ask a question

`ask()` runs the whole pipeline and prints each stage, so you can see exactly how the answer was produced.

In [23]:
def ask(question: str, show_rules: bool = False) -> str:
    rules = retrieve_rules(question)
    if show_rules:
        print("RULES RETRIEVED:\n" + rules + "\n")

    sql, results = generate_and_run_sql(question, rules)
    print("SQL:\n" + sql + "\n")
    print("RESULTS:\n" + (results[:1500] if results else "(no rows)") + "\n")

    answer = llm.invoke(EXPLAIN_PROMPT.format(
        dates=date_context(question),
        question=question,
        sql=sql,
        results=results[:4000] if results else "(no rows)",
    )).content

    print("=" * 70)
    print(answer)
    return answer

## 8. Example questions

### Revenue for premium customers, last quarter vs previous quarter

In [24]:
ask("What is the total revenue for premium customers last quarter? "
    "Compare it with the previous quarter.", show_rules=True);

RULES RETRIEVED:
BUSINESS RULE: PREMIUM CUSTOMERS
A premium customer is a customer whose lifetime revenue (all Completed orders, any date) is greater than 10,000. Calculate it with GROUP BY o.CustomerID HAVING SUM(oi.Quantity * oi.UnitPrice * (1 - oi.Discount)) > 10000. Premium status is based on lifetime spend, not spend within the period being analysed.

METRIC: QUARTER-ON-QUARTER GROWTH
Growth % = (revenue this quarter - revenue previous quarter) / revenue previous quarter * 100.

T-SQL TIP: START OF LAST QUARTER
Start of the current quarter: DATEADD(QUARTER, DATEDIFF(QUARTER, 0, GETDATE()), 0). Start of last quarter: DATEADD(QUARTER, DATEDIFF(QUARTER, 0, GETDATE()) - 1, 0). Start of the previous quarter: DATEADD(QUARTER, DATEDIFF(QUARTER, 0, GETDATE()) - 2, 0). Filter with OrderDate >= start AND OrderDate < next start.

METRIC: AVERAGE ORDER VALUE
Average order value = total revenue divided by COUNT(DISTINCT o.OrderID), Completed orders only.

BUSINESS RULE: QUARTERS
Quarters are c

### Highest-revenue year

In [25]:
ask("Which year had the highest revenue?");

SQL:
SELECT TOP 1 YEAR(o.OrderDate) AS [Year],
       SUM(oi.Quantity * oi.UnitPrice * (1 - oi.Discount)) AS Revenue
FROM dbo.Orders o
JOIN dbo.OrderItems oi ON oi.OrderID = o.OrderID
WHERE o.Status = 'Completed'
GROUP BY YEAR(o.OrderDate)
ORDER BY Revenue DESC

RESULTS:
[{'Year': 2025, 'Revenue': Decimal('891941.4340')}]

The highest revenue was achieved in the year 2025, with a total revenue of £891,941.43.


### Try your own

In [26]:
ask("What is the best-selling product of 2025?");

SQL:
SELECT TOP 1 p.ProductName, SUM(oi.Quantity) AS TotalQuantity
FROM dbo.OrderItems oi
JOIN dbo.Products p ON p.ProductID = oi.ProductID
JOIN dbo.Orders o ON o.OrderID = oi.OrderID
WHERE o.Status = 'Completed'
  AND YEAR(o.OrderDate) = 2025
GROUP BY p.ProductName
ORDER BY TotalQuantity DESC

RESULTS:
[{'ProductName': 'Printer Paper A4 (Box)', 'TotalQuantity': 1151}]

The best-selling product in 2025 was 'Printer Paper A4 (Box)', with a total quantity of 1151 units sold.


## 9. Validate against a hand-written query

LLMs can write SQL that runs but answers the wrong question. Examples include forgetting a discount, using the wrong date range, or defining premium customers by the wrong period.

This cell runs the correct premium-revenue query directly, so you can compare it with the answer in section 8.

In [37]:
ground_truth_sql = """
WITH premium AS (
    SELECT o.CustomerID
    FROM dbo.Orders o
    JOIN dbo.OrderItems oi ON oi.OrderID = o.OrderID
    WHERE o.Status = 'Completed'
    GROUP BY o.CustomerID
    HAVING SUM(oi.Quantity * oi.UnitPrice * (1 - oi.Discount)) > 10000
)
SELECT
    YEAR(o.OrderDate)              AS [Year],
    DATEPART(QUARTER, o.OrderDate) AS [Quarter],
    CAST(SUM(oi.Quantity * oi.UnitPrice * (1 - oi.Discount)) AS DECIMAL(12,2)) AS PremiumRevenue
FROM dbo.Orders o
JOIN dbo.OrderItems oi ON oi.OrderID = o.OrderID
JOIN premium p         ON p.CustomerID = o.CustomerID
WHERE o.Status = 'Completed'
  AND o.OrderDate >= DATEADD(QUARTER, DATEDIFF(QUARTER, 0, GETDATE()) - 2, 0)
  AND o.OrderDate <  DATEADD(QUARTER, DATEDIFF(QUARTER, 0, GETDATE()), 0)
GROUP BY YEAR(o.OrderDate), DATEPART(QUARTER, o.OrderDate)
ORDER BY [Year], [Quarter];
"""
print(db.run(ground_truth_sql, include_columns=True))

[{'Year': 2026, 'Quarter': 2, 'PremiumRevenue': Decimal('207659.76')}, {'Year': 2026, 'Quarter': 3, 'PremiumRevenue': Decimal('186038.94')}]


## Limitations and next steps

**Known limitations**

- **Small models make subtle SQL mistakes.** A 7B model sometimes drops part of a business rule even when it's retrieved. Always check the SQL that `ask()` prints. A larger model (e.g. `qwen2.5-coder:14b`, about 9 GB) is noticeably more accurate if your hardware allows.
- **Retrieval isn't perfect.** If a rule isn't in the top results, the model works without it. Use `show_rules=True` to see what was retrieved.
- **The safety check is basic.** It blocks obvious write statements, but you should still connect with a **read-only** login and never point this at a production database.
- **One question, one query.** Questions that need several separate queries may come back incomplete.
- **The demo data is randomly generated**, so figures change each time `create_TextToSQLDemo.sql` is re-run.

**Ideas for extending it**

- Add few-shot examples (question → correct SQL) to the data dictionary.
- Add a simple web front end with Streamlit or Gradio.
- Build an evaluation set of questions with known answers and compare accuracy across models.